## Imports

In [1]:
import os
import json
import random
import shutil
import tempfile

import torch

from huggingface_hub import login, HfApi, hf_hub_download
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
!pip install -q tokenizers 
!pip install ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 2.7 MB/s eta 0:00:00


In [3]:
!pip install -q bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.2 MB/s eta 0:00:00


In [4]:
import re
import ijson
from collections import Counter

from tokenizers import ByteLevelBPETokenizer

## Configuration


In [5]:
# =========================
# PROJECT CONFIGURATION
# =========================

# Hugging Face
HF_USERNAME = "A-240031"
HF_REPO_NAME = "legal-lm-phase1"
HF_MODEL_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"
HF_DATASET_REPO_ID = "Exploration-Lab/CS-781-Capstone"
# Dataset files
DATASET_FILES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

# Checkpointing
CHECKPOINT_INTERVAL_TOKENS = 1_000_000

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


#TOKENIZER CONFIGURATION
VOCAB_SIZE = 32_768
MIN_FREQUENCY = 5
CUSTOM_TOKEN_MIN_FREQUENCY = 100

SPECIAL_TOKENS = [
    "<|pad|>",
    "<|unk|>",
    "<|bos|>",
    "<|eos|>",
]

ARCH_TRAIN_TOKENS = 50_000_000
VAL_EXAMPLES = 500
VAL_PROMPT_WORDS = 512
VAL_TARGET_WORDS = 128
VAL_DOC_FRACTION = 0.01
SEED = 42

ARCH_TRAIN_PATH = "/kaggle/working/architecture_data/arch_train_200M.bin"
ARCH_VAL_PATH = "/kaggle/working/architecture_data/arch_val_200M.json"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# =========================
# TRAINING CONFIGURATION
# =========================


In [6]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [7]:
BUCKET_NAME = "A-240031/legal-lm-phase1-bucket"
BUCKET_ROOT = f"hf://buckets/{BUCKET_NAME}"

from huggingface_hub import HfFileSystem

fs = HfFileSystem()

In [8]:
def upload_to_bucket(local_path, remote_path):
    remote_path = remote_path.lstrip("/")

    fs.put_file(
        local_path,
        f"{BUCKET_ROOT}/{remote_path}",
    )

    print(f"Uploaded: {remote_path}")

In [9]:
def download_from_bucket(remote_path, local_path):
    remote_path = remote_path.lstrip("/")

    fs.get_file(
        f"{BUCKET_ROOT}/{remote_path}",
        local_path,
    )

    print(f"Downloaded: {remote_path} -> {local_path}")

## Checkpoint related functions

In [10]:
def upload_checkpoint(tmp_dir, tag):

    remote_dir = f"checkpoints/{tag}"

    for filename in os.listdir(tmp_dir):
        local_path = os.path.join(tmp_dir, filename)

        upload_to_bucket(
            local_path,
            f"{remote_dir}/{filename}",
        )

In [11]:
def save_checkpoint(tag, model, optimizer, scheduler=None, scaler=None,
                    tokens_seen=0, global_step=0, epoch=0, extra_state=None):

    with tempfile.TemporaryDirectory() as tmp_dir:

        torch.save(model.state_dict(), os.path.join(tmp_dir, "model.pt"))
        torch.save(optimizer.state_dict(), os.path.join(tmp_dir, "optimizer.pt"))

        if scheduler is not None:
            torch.save(scheduler.state_dict(), os.path.join(tmp_dir, "scheduler.pt"))

        if scaler is not None:
            torch.save(scaler.state_dict(), os.path.join(tmp_dir, "scaler.pt"))

        rng_state = {"python": random.getstate(), "torch": torch.get_rng_state()}

        if torch.cuda.is_available():
            rng_state["cuda"] = torch.cuda.get_rng_state_all()

        torch.save(rng_state, os.path.join(tmp_dir, "rng_state.pt"))

        state = {
            "tag": tag,
            "tokens_seen": tokens_seen,
            "global_step": global_step,
            "epoch": epoch,
            "extra_state": extra_state,
        }

        with open(os.path.join(tmp_dir, "training_state.json"), "w") as f:
            json.dump(state, f, indent=2)

        upload_checkpoint(tmp_dir,tag)

def load_checkpoint(tag, model, optimizer, scheduler=None, scaler=None):

    with tempfile.TemporaryDirectory() as tmp_dir:

        files = [
            "model.pt",
            "optimizer.pt",
            "scheduler.pt",
            "scaler.pt",
            "rng_state.pt",
            "training_state.json",
        ]

        checkpoint_dir = os.path.join(
            tmp_dir,
            "checkpoints",
            tag,
        )

        for filename in files:
            try:
                download_from_bucket(
                    f"checkpoints/{tag}/{filename}",
                    os.path.join(checkpoint_dir, filename),
                )
            except Exception:
                pass

        model.load_state_dict(
            torch.load(
                os.path.join(checkpoint_dir, "model.pt"),
                map_location=DEVICE,
                weights_only=True,
            )
        )

        optimizer.load_state_dict(
            torch.load(
                os.path.join(checkpoint_dir, "optimizer.pt"),
                map_location=DEVICE,
                weights_only=False,
            )
        )

        scheduler_path = os.path.join(checkpoint_dir, "scheduler.pt")

        if scheduler is not None and os.path.exists(scheduler_path):
            scheduler.load_state_dict(
                torch.load(
                    scheduler_path,
                    map_location=DEVICE,
                    weights_only=False,
                )
            )

        scaler_path = os.path.join(checkpoint_dir, "scaler.pt")

        if scaler is not None and os.path.exists(scaler_path):
            scaler.load_state_dict(
                torch.load(
                    scaler_path,
                    map_location=DEVICE,
                    weights_only=False,
                )
            )

        rng_state = torch.load(
            os.path.join(checkpoint_dir, "rng_state.pt"),
            map_location="cpu",
            weights_only=False,
        )

        random.setstate(rng_state["python"])
        torch.set_rng_state(rng_state["torch"])

        if torch.cuda.is_available() and "cuda" in rng_state:
            torch.cuda.set_rng_state_all(rng_state["cuda"])

        with open(
            os.path.join(checkpoint_dir, "training_state.json")
        ) as f:
            state = json.load(f)

    return state

## Loading training data

In [12]:
from pathlib import Path
from tokenizers import Tokenizer

LOCAL_DIR = Path("tokenizer_126M_decoder")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_TOKENIZER = LOCAL_DIR / "tokenizer.json"

# Download exact tokenizer from bucket
download_from_bucket(
    "tokenizer_126M_decoder/tokenizer.json",
    str(LOCAL_TOKENIZER)
)

# Load tokenizer
tokenizer = Tokenizer.from_file(
    str(LOCAL_TOKENIZER)
)

print("Tokenizer loaded successfully")
print("Vocab size:", tokenizer.get_vocab_size())

(…)et/tokenizer_126M_decoder/tokenizer.json:   0%|          | 0.00/2.47M [00:00<?, ?B/s]

Downloaded: tokenizer_126M_decoder/tokenizer.json -> tokenizer_126M_decoder/tokenizer.json
Tokenizer loaded successfully
Vocab size: 33467


In [13]:
import json
from huggingface_hub import hf_hub_download

HF_DATASET_REPO_ID = "Exploration-Lab/CS-781-Capstone"

# --------------------------------------------------
# Download
# --------------------------------------------------

fold1_text_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Train/fold1-text.json",
    repo_type="dataset"
)

fold1_gold_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Train/fold1-gold.json",
    repo_type="dataset"
)

fold2_text_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Train/fold2-text.json",
    repo_type="dataset"
)

fold2_gold_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Train/fold2-gold.json",
    repo_type="dataset"
)

labels_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Train/labels.txt",
    repo_type="dataset"
)

# --------------------------------------------------
# Load JSON
# --------------------------------------------------

with open(fold1_text_path, "r", encoding="utf-8") as f:
    fold1_text = json.load(f)

with open(fold1_gold_path, "r", encoding="utf-8") as f:
    fold1_gold = json.load(f)

with open(fold2_text_path, "r", encoding="utf-8") as f:
    fold2_text = json.load(f)

with open(fold2_gold_path, "r", encoding="utf-8") as f:
    fold2_gold = json.load(f)

# --------------------------------------------------
# Load labels
# --------------------------------------------------

with open(labels_path, "r", encoding="utf-8") as f:
    labels = [line.strip() for line in f if line.strip()]

print("Fold 1 documents:", len(fold1_text))
print("Fold 1 annotations:", len(fold1_gold))

print("Fold 2 documents:", len(fold2_text))
print("Fold 2 annotations:", len(fold2_gold))

print("Labels:")
print(labels)

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Fold 1 documents: 35
Fold 1 annotations: 35
Fold 2 documents: 35
Fold 2 annotations: 35
Labels:
['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']


In [14]:
VOCAB_SIZE = tokenizer.get_vocab_size()

BATCH_SIZE = 4

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0

ARCH_TRAIN_TOKENS = 100_000_000
CHECKPOINT_INTERVAL_TOKENS = 1_000_000

In [15]:
VOCAB_SIZE

33467

## Model - A1 

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=512, base=10000):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (torch.arange(0, head_dim, 2).float() / head_dim)
        )

        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)

        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, q, k):
        # q, k: [B, H, T, D]
        T = q.size(2)

        cos = self.cos[:T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[:T].unsqueeze(0).unsqueeze(0)

        q1, q2 = q[..., ::2], q[..., 1::2]
        k1, k2 = k[..., ::2], k[..., 1::2]

        q_rot = torch.stack(
            [q1 * cos - q2 * sin,
             q1 * sin + q2 * cos],
            dim=-1,
        ).flatten(-2)

        k_rot = torch.stack(
            [k1 * cos - k2 * sin,
             k1 * sin + k2 * cos],
            dim=-1,
        ).flatten(-2)

        return q_rot, k_rot


# ============================================================
# SwiGLU
# ============================================================

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(
            F.silu(self.gate_proj(x)) * self.up_proj(x)
        )


# ============================================================
# GQA + SDPA
# ============================================================

class GQAAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()

        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(
            d_model,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False,
        )

        self.rope = RotaryEmbedding(
            self.head_dim,
            max_seq_len=512,
        )

    def forward(self, x):

        B, T, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B, T,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        q, k = self.rope(q, k)

        # Repeat KV heads for GQA
        repeat_factor = self.n_heads // self.n_kv_heads

        k = k.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        v = v.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        # PyTorch SDPA can dispatch to Flash Attention
        # / memory-efficient kernels automatically.
        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True,
        )

        attn_out = (
            attn_out
            .transpose(1, 2)
            .contiguous()
            .view(B, T, -1)
        )

        return self.out_proj(attn_out)


# ============================================================
# Decoder Block
# ============================================================

class DecoderBlock(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        d_ff,
    ):
        super().__init__()

        self.norm1 = nn.RMSNorm(d_model)
        self.attention = GQAAttention(
            d_model,
            n_heads,
            n_kv_heads,
        )

        self.norm2 = nn.RMSNorm(d_model)
        self.ffn = SwiGLU(
            d_model,
            d_ff,
        )

    def forward(self, x):

        x = x + self.attention(
            self.norm1(x)
        )

        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ============================================================
# Decoder-only Language Model
# ============================================================

class EfficientDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=768,
        n_heads=12,
        n_kv_heads=4,
        n_layers=12,
        d_ff=3072,
        max_seq_len=512,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model

        self.token_embedding = nn.Embedding(                             
            vocab_size,
            d_model,
        )

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                d_ff=d_ff,
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.RMSNorm(d_model)

        # Tied LM head
        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False,
        )

        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.Embedding):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.RMSNorm):
            nn.init.ones_(module.weight)

    def forward(self, input_ids):

        x = self.token_embedding(input_ids)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        return self.lm_head(x)

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class LormaLinearPi(nn.Module):
    def __init__(
        self,
        original_linear: nn.Linear,
        lorma_r: int = 16,
        lorma_alpha: int = 32,
        mode: str = "pre",
        do_pi: bool = True,
        lorma_dropout: float = 0.05,
    ):
        super().__init__()

        assert mode in [
            "pre", "post", "complete_post", "complete_pre"
        ]

        self.mode = mode
        self.original_layer = original_linear
        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features
        self.lorma_r = lorma_r
        self.scaling = lorma_alpha / lorma_r
        self.do_pi = do_pi

        self.lora_dropout = (
            nn.Dropout(lorma_dropout)
            if lorma_dropout > 0
            else nn.Identity()
        )

        if mode == "pre":
            self.lora_B = nn.Parameter(
                original_linear.weight.new_zeros(
                    self.out_features, lorma_r
                )
            )
            self.lora_A = nn.Parameter(
                original_linear.weight.new_zeros(
                    lorma_r, self.out_features
                )
            )

        elif mode == "post":
            self.lora_B = nn.Parameter(
                original_linear.weight.new_zeros(
                    self.in_features, lorma_r
                )
            )
            self.lora_A = nn.Parameter(
                original_linear.weight.new_zeros(
                    lorma_r, self.in_features
                )
            )

        elif mode == "complete_post":
            self.lora_A = nn.Parameter(
                original_linear.weight.new_zeros(
                    self.in_features, self.in_features
                )
            )

        else:  # complete_pre
            self.lora_A = nn.Parameter(
                original_linear.weight.new_zeros(
                    self.out_features, self.out_features
                )
            )

        if mode in ("complete_post", "complete_pre"):
            nn.init.eye_(self.lora_A)

        elif not do_pi:
            nn.init.kaiming_uniform_(
                self.lora_B, a=math.sqrt(5)
            )

            with torch.no_grad():
                self.lora_A.copy_(
                    torch.linalg.pinv(self.lora_B)
                )
                self.lora_A.div_(math.sqrt(self.scaling))
                self.lora_B.div_(math.sqrt(self.scaling))

        else:
            with torch.no_grad():
                nn.init.kaiming_uniform_(
                    self.lora_B, a=math.sqrt(5)
                )
                self.lora_B[:, 0] = 1.0

                self.lora_A.zero_()
                self.lora_A[0, 0] = 1.0

                self.lora_A.div_(math.sqrt(self.scaling))
                self.lora_B.div_(math.sqrt(self.scaling))

        if self.do_pi:
            self.shifting_matrices = {}

    def shift(self, M):
        # Preserve the shift operation from the supplied implementation.
        shape = M.shape
        device = M.device
        key = (shape, device)

        if key not in self.shifting_matrices:
            n_rows, n_cols = shape

            row_indices = torch.arange(
                n_rows, device=device
            ).view(-1, 1)

            col_indices = torch.arange(
                n_cols, device=device
            ).view(1, -1)

            indices = (col_indices - row_indices) % n_cols
            self.shifting_matrices[key] = indices

        return torch.gather(
            M, 1, self.shifting_matrices[key]
        )

    def forward(self, x):
        W = self.original_layer.weight

        if self.mode == "pre":
            if self.do_pi:
                adapted_weight = (
                    self.shift(self.lora_B @ self.lora_A) @ W
                )
            else:
                adapted_weight = (
                    self.lora_B @ (self.lora_A @ W)
                )

            output = F.linear(
                self.lora_dropout(x),
                adapted_weight * self.scaling,
                self.original_layer.bias,
            )

        elif self.mode == "post":
            if self.do_pi:
                adapted_weight = (
                    W @ self.shift(self.lora_B @ self.lora_A)
                )
            else:
                adapted_weight = (
                    (W @ self.lora_B) @ self.lora_A
                )

            output = F.linear(
                self.lora_dropout(x),
                adapted_weight * self.scaling,
                self.original_layer.bias,
            )

        elif self.mode == "complete_post":
            adapted_weight = W @ self.lora_A
            output = F.linear(
                self.lora_dropout(x),
                adapted_weight,
                self.original_layer.bias,
            )

        else:
            adapted_weight = self.lora_A @ W
            output = F.linear(
                self.lora_dropout(x),
                adapted_weight,
                self.original_layer.bias,
            )

        return output

class EfficientDecoderLMForNER(EfficientDecoderLM):

    def __init__(
        self,
        vocab_size,
        num_labels=25,
        d_model=768,
        n_heads=12,
        n_kv_heads=4,
        n_layers=12,
        d_ff=3072,
        max_seq_len=512,
    ):
        super().__init__(
            vocab_size=vocab_size,
            d_model=d_model,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            n_layers=n_layers,
            d_ff=d_ff,
            max_seq_len=max_seq_len,
        )

        self.ner_head = nn.Linear(d_model, num_labels)

    def forward(self, input_ids, word_groups):
    # input_ids: [B, T]
        B = input_ids.size(0)
    
        x = self.token_embedding(input_ids)
    
        for block in self.blocks:
            x = block(x)
    
        x = self.final_norm(x)
    
        batch_word_hidden = []
        max_words = 0
    
        for b in range(B):
    
            word_hidden = []
    
            for group in word_groups[b]:
                h = x[b, group].mean(dim=0)
                word_hidden.append(h)
    
            word_hidden = torch.stack(word_hidden)
    
            batch_word_hidden.append(word_hidden)
    
            max_words = max(
                max_words,
                word_hidden.size(0)
            )
    
        # [B, max_words, D]
        padded_word_hidden = x.new_zeros(
            B,
            max_words,
            x.size(-1)
        )
    
        for b, word_hidden in enumerate(batch_word_hidden):
    
            n_words = word_hidden.size(0)
    
            padded_word_hidden[
                b, :n_words
            ] = word_hidden
    
        # [B, max_words, num_labels]
        logits = self.ner_head(padded_word_hidden)
    
        return logits

In [17]:
d_model    = 1024
n_layers   = 8
n_heads    = 8
n_kv_heads = 4        # GQA
d_ff       = 2816     # SwiGLU
seq_len    = 512

In [18]:
teacher_base = EfficientDecoderLM(
    vocab_size=VOCAB_SIZE,
    d_model=d_model,
    n_heads=n_heads,
    n_kv_heads=n_kv_heads,
    n_layers=n_layers,
    d_ff=d_ff,
    max_seq_len=seq_len,
).to(DEVICE)
print(
    f"Parameters: " 
    f"{sum(p.numel() for p in teacher_base.parameters()) / 1e6:.2f}M")

Parameters: 128.66M


In [19]:
teacher_optimizer_dummy = torch.optim.AdamW(
    teacher_base.parameters(),
    lr=LEARNING_RATE,
)

teacher_state = load_checkpoint(
    "decoder_126M",
    teacher_base,
    teacher_optimizer_dummy,
)

print("Loaded Phase-1 decoder checkpoint")
print(teacher_state)

(…)bucket/checkpoints/decoder_126M/model.pt:   0%|          | 0.00/515M [00:00<?, ?B/s]

Downloaded: checkpoints/decoder_126M/model.pt -> /tmp/tmpj9bbiddu/checkpoints/decoder_126M/model.pt


(…)et/checkpoints/decoder_126M/optimizer.pt:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

Downloaded: checkpoints/decoder_126M/optimizer.pt -> /tmp/tmpj9bbiddu/checkpoints/decoder_126M/optimizer.pt


(…)et/checkpoints/decoder_126M/rng_state.pt:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Downloaded: checkpoints/decoder_126M/rng_state.pt -> /tmp/tmpj9bbiddu/checkpoints/decoder_126M/rng_state.pt


(…)kpoints/decoder_126M/training_state.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Downloaded: checkpoints/decoder_126M/training_state.json -> /tmp/tmpj9bbiddu/checkpoints/decoder_126M/training_state.json
Loaded Phase-1 decoder checkpoint
{'tag': 'decoder_126M', 'tokens_seen': 200007680, 'global_step': 24415, 'epoch': 0, 'extra_state': None}


In [20]:
del teacher_optimizer_dummy
del teacher_state
import gc
gc.collect()

167

In [21]:
labels

['APP',
 'RESP',
 'A.COUNSEL',
 'R.COUNSEL',
 'JUDGE',
 'WIT',
 'AUTH',
 'COURT',
 'STAT',
 'PREC',
 'DATE',
 'CASENO']

In [22]:
teacher = EfficientDecoderLMForNER(
    vocab_size=VOCAB_SIZE,
    num_labels=25,
    d_model=d_model,
    n_heads=n_heads,
    n_kv_heads=n_kv_heads,
    n_layers=n_layers,
    d_ff=d_ff,
    max_seq_len=seq_len,
).to(DEVICE)

In [23]:
CHECKPOINT_PATH = "/kaggle/working/teacher_base.pt"

torch.save(
    teacher_base.state_dict(),
    CHECKPOINT_PATH
)
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True,
)

missing, unexpected = teacher.load_state_dict(
    checkpoint,
    strict=False,
)

print("Missing:", missing)
print("Unexpected:", unexpected)

Missing: ['ner_head.weight', 'ner_head.bias']
Unexpected: []


In [24]:
for block in teacher.blocks:
    attn = block.attention

    attn.q_proj = LormaLinearPi(
        attn.q_proj, lorma_r=16, lorma_alpha=32,
        mode="pre", do_pi=True, lorma_dropout=0.05
    )
    attn.k_proj = LormaLinearPi(
        attn.k_proj, lorma_r=16, lorma_alpha=32,
        mode="pre", do_pi=True, lorma_dropout=0.05
    )
    attn.v_proj = LormaLinearPi(
        attn.v_proj, lorma_r=16, lorma_alpha=32,
        mode="pre", do_pi=True, lorma_dropout=0.05
    )
    attn.out_proj = LormaLinearPi(
        attn.out_proj, lorma_r=16, lorma_alpha=32,
        mode="pre", do_pi=True, lorma_dropout=0.05
    )

In [25]:
for p in teacher.parameters():
    p.requires_grad = False

In [26]:
for block in teacher.blocks:
    attn = block.attention

    for proj in [attn.q_proj, attn.k_proj, attn.v_proj, attn.out_proj]:
        proj.lora_A.requires_grad = True
        proj.lora_B.requires_grad = True

# Unfreeze both hierarchical NER heads
for p in teacher.ner_head.parameters():
    p.requires_grad = True

In [27]:
# Entity labels from labels.txt
print(labels)

BIO_LABELS = ["O"]

for entity in labels:
    BIO_LABELS.append(f"B-{entity}")
    BIO_LABELS.append(f"I-{entity}")

label2id = {
    label: i
    for i, label in enumerate(BIO_LABELS)
}

id2label = {
    i: label
    for label, i in label2id.items()
}

print("Number of BIO labels:", len(BIO_LABELS))
print(label2id)

['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Number of BIO labels: 25
{'O': 0, 'B-APP': 1, 'I-APP': 2, 'B-RESP': 3, 'I-RESP': 4, 'B-A.COUNSEL': 5, 'I-A.COUNSEL': 6, 'B-R.COUNSEL': 7, 'I-R.COUNSEL': 8, 'B-JUDGE': 9, 'I-JUDGE': 10, 'B-WIT': 11, 'I-WIT': 12, 'B-AUTH': 13, 'I-AUTH': 14, 'B-COURT': 15, 'I-COURT': 16, 'B-STAT': 17, 'I-STAT': 18, 'B-PREC': 19, 'I-PREC': 20, 'B-DATE': 21, 'I-DATE': 22, 'B-CASENO': 23, 'I-CASENO': 24}


In [28]:
def prepare_lner_data(text_data, gold_data, tokenizer):

    processed_data = {}

    for doc_id, text in text_data.items():

       

        # --------------------------------------------------
        # 1. Tokenize
        # --------------------------------------------------

        encoding = tokenizer.encode(text)

        tokens = encoding.tokens
        input_ids = encoding.ids
        offsets = encoding.offsets

        # --------------------------------------------------
        # 2. Group tokens
        #
        # Rule:
        #   Ġ  -> new word
        #   no Ġ -> continue previous word
        # --------------------------------------------------

        word_groups = []
        current_group = []

        for i, token in enumerate(tokens):

            start, end = offsets[i]

            if start == end:
                continue

            if current_group and token.startswith("Ġ"):
                word_groups.append(current_group)
                current_group = []

            current_group.append(i)

        if current_group:
            word_groups.append(current_group)

        # --------------------------------------------------
        # 3. Correct character spans
        #
        # Raw tokenizer offsets include the whitespace
        # represented by Ġ.
        # Remove that whitespace from the beginning.
        # --------------------------------------------------

        word_spans = []

        for group in word_groups:

            start = offsets[group[0]][0]
            end = offsets[group[-1]][1]

            while start < end and text[start].isspace():
                start += 1

            word_spans.append((start, end))

        # --------------------------------------------------
        # 4. Initialize labels
        # --------------------------------------------------

        word_labels = ["O"] * len(word_groups)

        # --------------------------------------------------
        # 5. Align gold entities
        # --------------------------------------------------
        if gold_data is not None:
            gold_entities = gold_data[doc_id]
            for entity_start, entity_end, entity_type, entity_text in gold_entities:
    
                overlapping_words = []
    
                for word_idx, (word_start, word_end) in enumerate(word_spans):
    
                    if word_start < entity_end and word_end > entity_start:
                        overlapping_words.append(word_idx)
    
                if not overlapping_words:
                    continue
    
                word_labels[overlapping_words[0]] = f"B-{entity_type}"
    
                for word_idx in overlapping_words[1:]:
                    word_labels[word_idx] = f"I-{entity_type}"

        # --------------------------------------------------
        # 6. Save
        # --------------------------------------------------

        processed_data[doc_id] = {
            "input_ids": input_ids,
            "offsets": offsets,
            "word_groups": word_groups,
            "word_spans": word_spans,
            "word_labels": word_labels,
        }

    return processed_data

In [29]:
fold1_processed = prepare_lner_data(
    fold1_text,
    fold1_gold,
    tokenizer
)


In [30]:
fold2_processed = prepare_lner_data(
    fold2_text,
    fold2_gold,
    tokenizer
)

In [31]:
MAX_SEQ_LEN = 512

def make_chunks(example, max_seq_len=512):
    """
    Split one processed document into chunks at word boundaries.

    Returns list of:
        {
            "input_ids": [...],
            "word_groups": [...],
            "word_labels": [...]
        }
    """

    input_ids = example["input_ids"]
    word_groups = example["word_groups"]
    word_labels = example["word_labels"]
    word_spans = example["word_spans"]
    chunks = []

    current_words = []
    current_start = None
    current_end = None

    for word_idx, group in enumerate(word_groups):

        word_start = group[0]
        word_end = group[-1] + 1

        if current_start is None:
            current_start = word_start
            current_end = word_end
            current_words = [word_idx]
            continue

        # Would adding this word exceed 2048?
        new_end = word_end

        if new_end - current_start > max_seq_len:

            # Save current chunk
            chunk_input_ids = input_ids[current_start:current_end]

            chunk_groups = [
                [
                    token_idx - current_start
                    for token_idx in word_groups[w]
                ]
                for w in current_words
            ]

            chunk_labels = [
                word_labels[w]
                for w in current_words
            ]
            chunk_spans = [
                word_spans[w]
                for w in current_words
            ]
            chunks.append({
                "input_ids": chunk_input_ids,
                "word_groups": chunk_groups,
                "word_labels": chunk_labels,
                "word_spans" : chunk_spans
            })

            # Start new chunk
            current_start = word_start
            current_end = word_end
            current_words = [word_idx]

        else:
            current_end = word_end
            current_words.append(word_idx)

    # Last chunk
    if current_words:

        chunk_input_ids = input_ids[current_start:current_end]

        chunk_groups = [
            [
                token_idx - current_start
                for token_idx in word_groups[w]
            ]
            for w in current_words
        ]

        chunk_labels = [
            word_labels[w]
            for w in current_words
        ]
        chunk_spans = [
            word_spans[w]
            for w in current_words
        ]
        chunks.append({
            "input_ids": chunk_input_ids,
            "word_groups": chunk_groups,
            "word_labels": chunk_labels,
            "word_spans" : chunk_spans
        })

    return chunks

In [32]:
import random

SEED = 42
TRAIN_RATIO = 0.99
MAX_SEQ_LEN = 512

# --------------------------------------------------
# 1. Combine Fold 1 + Fold 2
# --------------------------------------------------

all_processed_data = {}

all_processed_data.update(fold1_processed)
all_processed_data.update(fold2_processed)

doc_ids = list(all_processed_data.keys())

print("Total documents:", len(doc_ids))

# --------------------------------------------------
# 2. Shuffle documents
# --------------------------------------------------

rng = random.Random(SEED)
rng.shuffle(doc_ids)

# --------------------------------------------------
# 3. 99/1 document-level split
# --------------------------------------------------

split_idx = int(TRAIN_RATIO * len(doc_ids))

train_doc_ids = doc_ids[:split_idx]
val_doc_ids = doc_ids[split_idx:]

print("Train documents:", len(train_doc_ids))
print("Validation documents:", len(val_doc_ids))

print(
    "Train ratio:",
    len(train_doc_ids) / len(doc_ids)
)

print(
    "Validation ratio:",
    len(val_doc_ids) / len(doc_ids)
)

# --------------------------------------------------
# 4. Chunk training documents
# --------------------------------------------------

train_chunks = []

for doc_id in train_doc_ids:

    chunks = make_chunks(
        all_processed_data[doc_id],
        MAX_SEQ_LEN
    )

    train_chunks.extend(chunks)

# --------------------------------------------------
# 5. Chunk validation documents
# --------------------------------------------------

val_chunks = []

for doc_id in val_doc_ids:

    chunks = make_chunks(
        all_processed_data[doc_id],
        MAX_SEQ_LEN
    )

    val_chunks.extend(chunks)

# --------------------------------------------------
# 6. Check
# --------------------------------------------------

print("Train chunks:", len(train_chunks))
print("Validation chunks:", len(val_chunks))

assert len(train_chunks) > 0
assert len(val_chunks) > 0

assert all(
    len(chunk["input_ids"]) <= MAX_SEQ_LEN
    for chunk in train_chunks
)

assert all(
    len(chunk["input_ids"]) <= MAX_SEQ_LEN
    for chunk in val_chunks
)

Total documents: 70
Train documents: 69
Validation documents: 1
Train ratio: 0.9857142857142858
Validation ratio: 0.014285714285714285
Train chunks: 911
Validation chunks: 15


In [33]:
def collate_lner(batch, pad_token_id):

    max_len = max(
        len(x["input_ids"])
        for x in batch
    )

    max_words = max(
        len(x["word_labels"])
        for x in batch
    )

    input_ids = []
    word_groups = []
    word_labels = []

    for x in batch:

        # -------------------------
        # Token padding
        # -------------------------

        ids = x["input_ids"]

        padded = ids + [pad_token_id] * (
            max_len - len(ids)
        )

        input_ids.append(padded)

        # -------------------------
        # Word groups
        # -------------------------

        word_groups.append(
            x["word_groups"]
        )

        # -------------------------
        # Word-label padding
        # -------------------------

        labels = [
            label2id[label]
            for label in x["word_labels"]
        ]

        labels += [-100] * (
            max_words - len(labels)
        )

        word_labels.append(labels)

    return {
        "input_ids": torch.tensor(
            input_ids,
            dtype=torch.long
        ),

        "word_groups": word_groups,

        "word_labels": torch.tensor(
            word_labels,
            dtype=torch.long
        ),
    }

In [34]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_chunks,
    batch_size=4,
    shuffle=True,
    collate_fn=lambda batch: collate_lner(
        batch,
        tokenizer.token_to_id("<|pad|>")
    )
)

val_loader = DataLoader(
    val_chunks,
    batch_size=1,
    shuffle=False,
    collate_fn=lambda batch: collate_lner(
        batch,
        tokenizer.token_to_id("<|pad|>")
    )
)

In [35]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
teacher = teacher.to(device)

NUM_EPOCHS = 5
LR = 2e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
GRAD_ACCUM_STEPS = 4

criterion = nn.CrossEntropyLoss()

trainable_params = [
    p for p in teacher.parameters()
    if p.requires_grad
]

optimizer = AdamW(
    trainable_params,
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

print(
    "Trainable parameters:",
    sum(p.numel() for p in trainable_params)
)

Trainable parameters: 812057


In [36]:
ENTITY_LABELS = [
    "APP",
    "RESP",
    "A.COUNSEL",
    "R.COUNSEL",
    "JUDGE",
    "WIT",
    "AUTH",
    "COURT",
    "STAT",
    "PREC",
    "DATE",
    "CASENO",
]


In [37]:
def decode_bio_predictions(
    predictions,
    word_spans,
):
    entities = []

    current_label = None
    current_start = None
    current_end = None

    for label, (start, end) in zip(
        predictions,
        word_spans
    ):

        # O
        if label == "O":

            if current_label is not None:
                entities.append([
                    current_start,
                    current_end,
                    current_label
                ])

                current_label = None
                current_start = None
                current_end = None

            continue

        # B-ENTITY
        if label.startswith("B-"):

            if current_label is not None:
                entities.append([
                    current_start,
                    current_end,
                    current_label
                ])

            current_label = label[2:]
            current_start = start
            current_end = end

            continue

        # I-ENTITY
        if label.startswith("I-"):

            entity_type = label[2:]

            # Continue current entity
            if current_label == entity_type:

                current_end = end

            # Invalid I-ENTITY -> treat as a new entity
            else:

                if current_label is not None:
                    entities.append([
                        current_start,
                        current_end,
                        current_label
                    ])

                current_label = entity_type
                current_start = start
                current_end = end

    # Flush final entity
    if current_label is not None:
        entities.append([
            current_start,
            current_end,
            current_label
        ])

    return entities

In [38]:
def bio_to_spans(labels, word_spans):
    """
    Convert BIO word labels + character spans into:
        [start, end, label]
    """

    entities = []

    current_label = None
    current_start = None
    current_end = None

    for label, (start, end) in zip(
        labels,
        word_spans
    ):

        if label == "O":

            if current_label is not None:
                entities.append([
                    current_start,
                    current_end,
                    current_label
                ])

                current_label = None
                current_start = None
                current_end = None

            continue

        if label.startswith("B-"):

            if current_label is not None:
                entities.append([
                    current_start,
                    current_end,
                    current_label
                ])

            current_label = label[2:]
            current_start = start
            current_end = end

        elif label.startswith("I-"):

            entity_type = label[2:]

            if current_label == entity_type:

                current_end = end

            else:
                # Treat invalid I-X as a new entity
                if current_label is not None:
                    entities.append([
                        current_start,
                        current_end,
                        current_label
                    ])

                current_label = entity_type
                current_start = start
                current_end = end

    if current_label is not None:
        entities.append([
            current_start,
            current_end,
            current_label
        ])

    return entities

from tqdm import tqdm
import torch


def generate_val_predictions(
    teacher,
    all_processed_data,
    val_doc_ids,
    id2label,
    max_seq_len=512,
):
    teacher.eval()

    predicted = {}
    gold = {}

    with torch.no_grad():

        for doc_id in tqdm(
            val_doc_ids,
            desc="Validation prediction"
        ):

            example = all_processed_data[doc_id]

            chunks = make_chunks(
                example,
                max_seq_len
            )

            doc_predictions = []

            for chunk in chunks:

                input_ids = torch.tensor(
                    [chunk["input_ids"]],
                    dtype=torch.long,
                    device=DEVICE
                )

                word_groups = [
                    chunk["word_groups"]
                ]

                logits = teacher(
                    input_ids,
                    word_groups
                )

                predicted_ids = logits.argmax(
                    dim=-1
                )[0]

                predicted_labels = [
                    id2label[idx.item()]
                    for idx in predicted_ids
                ]

                entities = decode_bio_predictions(
                    predicted_labels,
                    chunk["word_spans"]
                )

                doc_predictions.extend(
                    entities
                )

            predicted[doc_id] = doc_predictions

            # Gold spans are obtained directly from the
            # original gold BIO labels + character spans
            gold_labels = example["word_labels"]

            gold[doc_id] = bio_to_spans(
                gold_labels,
                example["word_spans"]
            )

    return predicted, gold

from collections import defaultdict


def evaluate_strict_ner(
    predicted,
    gold,
    labels,
):
    """
    Strict entity-level evaluation.

    Returns:
        Macro-F1
        Macro-Precision
        Macro-Recall
        per-label statistics
    """

    stats = {
        label: {
            "tp": 0,
            "fp": 0,
            "fn": 0,
        }
        for label in labels
    }

    for doc_id in gold:

        gold_entities = gold.get(
            doc_id,
            []
        )

        pred_entities = predicted.get(
            doc_id,
            []
        )

        # Exact entity tuples
        gold_set = set(
            tuple(entity)
            for entity in gold_entities
        )

        pred_set = set(
            tuple(entity)
            for entity in pred_entities
        )

        # True positives
        for entity in pred_set & gold_set:

            label = entity[2]

            if label in stats:
                stats[label]["tp"] += 1

        # False positives
        for entity in pred_set - gold_set:

            label = entity[2]

            if label in stats:
                stats[label]["fp"] += 1

        # False negatives
        for entity in gold_set - pred_set:

            label = entity[2]

            if label in stats:
                stats[label]["fn"] += 1

    # -------------------------
    # Per-label metrics
    # -------------------------

    per_label = {}

    macro_precision = 0.0
    macro_recall = 0.0
    macro_f1 = 0.0

    for label in labels:

        tp = stats[label]["tp"]
        fp = stats[label]["fp"]
        fn = stats[label]["fn"]

        precision = (
            tp / (tp + fp)
            if tp + fp > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if tp + fn > 0
            else 0.0
        )

        f1 = (
            2 * precision * recall /
            (precision + recall)
            if precision + recall > 0
            else 0.0
        )

        per_label[label] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "tp": tp,
            "fp": fp,
            "fn": fn,
        }

        macro_precision += precision
        macro_recall += recall
        macro_f1 += f1

    n = len(labels)

    macro_precision /= n
    macro_recall /= n
    macro_f1 /= n

    return {
        "strict_macro_f1": macro_f1,
        "strict_macro_precision": macro_precision,
        "strict_macro_recall": macro_recall,
        "per_label": per_label,
    }

In [39]:
from collections import Counter
import torch
import torch.nn as nn

label_counts = Counter()

for chunk in train_chunks:
    for label in chunk["word_labels"]:
        label_counts[label2id[label]] += 1

counts = torch.tensor(
    [label_counts[i] for i in range(len(label2id))],
    dtype=torch.float
)

# Inverse square-root frequency to avoid excessively
# large weights for rare labels
weights = 1.0 / torch.sqrt(counts.clamp(min=1))

# Normalize so the average class weight is 1
weights = weights / weights.mean()

# Reduce the weight of O
weights[label2id["O"]] *= 0.25

# Normalize again
weights = weights / weights.mean()

class_weights = weights.to(DEVICE)



print({
    label: round(class_weights[idx].item(), 3)
    for label, idx in label2id.items()
})

{'O': 0.01, 'B-APP': 1.127, 'I-APP': 1.086, 'B-RESP': 1.164, 'I-RESP': 1.035, 'B-A.COUNSEL': 1.622, 'I-A.COUNSEL': 1.675, 'B-R.COUNSEL': 1.67, 'I-R.COUNSEL': 1.791, 'B-JUDGE': 1.453, 'I-JUDGE': 1.344, 'B-WIT': 1.657, 'I-WIT': 1.802, 'B-AUTH': 0.732, 'I-AUTH': 0.744, 'B-COURT': 0.72, 'I-COURT': 0.611, 'B-STAT': 0.65, 'I-STAT': 0.507, 'B-PREC': 0.659, 'I-PREC': 0.278, 'B-DATE': 0.581, 'I-DATE': 0.752, 'B-CASENO': 0.833, 'I-CASENO': 0.496}


In [40]:
def train_teacher(
    teacher,
    train_loader,
    val_loader,
    label2id,
    optimizer,
    trainable_params,
    tag,
    num_epochs=5,
    resume=True,
    resume_tag=None,
):
    device = next(teacher.parameters()).device

    criterion = nn.CrossEntropyLoss(
        ignore_index=-100
    )

    # =========================
    # RESUME
    # =========================

    start_epoch = 0

    if resume and resume_tag is not None:

        state = load_checkpoint(
            tag=resume_tag,
            model=teacher,
            optimizer=optimizer,
        )

        start_epoch = state["epoch"]

        print(
            f"Resuming from epoch {start_epoch}"
        )

    # =========================
    # TRAINING
    # =========================

    for epoch in range(start_epoch, num_epochs):

        teacher.train()

        total_train_loss = 0.0

        progress = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{num_epochs}"
        )

        for step, batch in enumerate(progress):

            input_ids = batch["input_ids"].to(device)

            word_groups = batch["word_groups"]

            targets = batch["word_labels"].to(device)

            optimizer.zero_grad(set_to_none=True)

            # [B, max_words, num_labels]
            logits = teacher(
                input_ids,
                word_groups
            )

            # Flatten:
            # logits  -> [B * max_words, num_labels]
            # targets -> [B * max_words]
            loss = criterion(
                logits.view(
                    -1,
                    logits.size(-1)
                ),
                targets.view(-1)
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                GRAD_CLIP
            )

            optimizer.step()

            total_train_loss += loss.item()

            progress.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        avg_train_loss = (
            total_train_loss /
            len(train_loader)
        )

        # =========================
        # VALIDATION
        # =========================

        teacher.eval()

        total_val_loss = 0.0
        
        with torch.no_grad():
        
            for batch in val_loader:
        
                input_ids = batch["input_ids"].to(device)
                word_groups = batch["word_groups"]
                targets = batch["word_labels"].to(device)
        
                logits = teacher(
                    input_ids,
                    word_groups
                )
        
                loss = criterion(
                    logits.view(-1, logits.size(-1)),
                    targets.view(-1)
                )
        
                total_val_loss += loss.item()
        
        avg_val_loss = (
            total_val_loss /
            len(val_loader)
        )
        
        # =========================
        # STRICT VALIDATION F1
        # =========================
        
        val_predicted, val_gold = generate_val_predictions(
            teacher=teacher,
            all_processed_data=all_processed_data,
            val_doc_ids=val_doc_ids,
            id2label=id2label,
            max_seq_len=MAX_SEQ_LEN,
        )
        
        val_metrics = evaluate_strict_ner(
            predicted=val_predicted,
            gold=val_gold,
            labels=ENTITY_LABELS,
        )
        
        val_f1 = val_metrics["strict_macro_f1"]
        # =========================
        # LOGGING
        # =========================

        print(
            f"\nEpoch {epoch + 1}/{num_epochs}"
        )

        print(
            f"Train Loss:    {avg_train_loss:.4f}"
        )

        print(
            f"Val Loss:      {avg_val_loss:.4f}"
        )

        print(
            f"Val Strict F1:  {val_f1:.4f}"
        )

        # =========================
        # SAVE
        # =========================

        save_checkpoint(
            tag=tag,
            model=teacher,
            optimizer=optimizer,
            epoch=epoch + 1,
            
        )

In [41]:
train_teacher(
    teacher=teacher,
    train_loader = train_loader,
    val_loader=val_loader,
    label2id=label2id,
    optimizer=optimizer,
    trainable_params=trainable_params,
    tag="teacher_lorma_unweighted",
    num_epochs=20,
)

Validation prediction: 100%|██████████| 1/1 [00:01<00:00,  1.08s/it]



Epoch 1/20
Train Loss:    0.2652
Val Loss:      0.1003
Val Strict F1:  0.2751


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded: checkpoints/teacher_lorma_unweighted/optimizer.pt


Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (270 outputs left out)

Uploaded: checkpoints/teacher_lorma_unweighted/rng_state.pt


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded: checkpoints/teacher_lorma_unweighted/model.pt


In [42]:
import json
import zipfile
import os
import torch
from tqdm import tqdm


def generate_predictions(
    teacher,
    processed_test,
    id2label,
    tag,
    max_seq_len=512,
):
    teacher.eval()

    predictions = {}

    with torch.no_grad():

        for doc_id, example in tqdm(
            processed_test.items(),
            desc="Generating predictions"
        ):

            chunks = make_chunks(
                example,
                max_seq_len
            )

            document_entities = []

            for chunk in chunks:

                input_ids = torch.tensor(
                    [chunk["input_ids"]],
                    dtype=torch.long,
                    device=DEVICE
                )

                word_groups = [
                    chunk["word_groups"]
                ]

                logits = teacher(
                    input_ids,
                    word_groups
                )

                predicted_ids = logits.argmax(
                    dim=-1
                )[0]

                predicted_labels = [
                    id2label[idx.item()]
                    for idx in predicted_ids
                ]

                entities = decode_bio_predictions(
                    predicted_labels,
                    chunk["word_spans"]
                )

                document_entities.extend(
                    entities
                )

            predictions[doc_id] = document_entities

    # =========================
    # Save predictions.json
    # =========================

    json_path = f"/kaggle/working/predictions.json"

    with open(json_path, "w") as f:
        json.dump(
            predictions,
            f,
            indent=2
        )

    # =========================
    # Zip as tag.zip
    # =========================

    zip_path = f"/kaggle/working/{tag}.zip"

    with zipfile.ZipFile(
        zip_path,
        "w",
        zipfile.ZIP_DEFLATED
    ) as z:

        z.write(
            json_path,
            arcname="prediction.json"
        )

    print(f"Saved: {zip_path}")

    return zip_path

In [43]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Test/fold3-text.json",
    repo_type="dataset",
)

with open(test_file_path, "r") as f:
    fold3_text = json.load(f)

print(f"Loaded {len(fold3_text)} test documents")
processed_test = prepare_lner_data(
    fold3_text,
    None,
    tokenizer
)

print(f"Processed {len(processed_test)} documents")

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Loaded 35 test documents
Processed 35 documents


In [44]:
generate_predictions(
    teacher,
    processed_test,
    id2label,
    "teacher_lorma_unweighted",
    max_seq_len=512,
)

Generating predictions: 100%|██████████| 35/35 [00:35<00:00,  1.01s/it]

Saved: /kaggle/working/teacher_lorma_unweighted.zip


'/kaggle/working/teacher_lorma_unweighted.zip'